# Qwen3-TTS LoRA Fine-Tuning — Malayalam

Fine-tunes `Qwen3-TTS-12Hz-1.7B-Base` with a LoRA adapter on the `siyah1/Malayalam-TTS-v2` dataset.

**Requirements:**  
- Run from the project root: `c:/Users/siyah/Pictures/ttsfine/`  
- Virtual environment must be active  
- PYTHONPATH must include `Qwen3-TTS/finetuning`

## 0. Environment Setup

In [ ]:
import sys
import os

# Make sure we are in the project root
PROJECT_ROOT = r"c:\Users\siyah\Pictures\ttsfine"
os.chdir(PROJECT_ROOT)

# Add the finetuning module to the path
FINETUNING_DIR = os.path.join(PROJECT_ROOT, "Qwen3-TTS", "finetuning")
if FINETUNING_DIR not in sys.path:
    sys.path.insert(0, FINETUNING_DIR)

print(f"Working directory : {os.getcwd()}")
print(f"Python path entry : {FINETUNING_DIR}")

## 1. Configuration

Edit the values below to match your setup.

In [ ]:
# ─── Paths ────────────────────────────────────────────────────────────────────
INIT_MODEL_PATH      = "Qwen3-TTS-Base"                              # local base model dir
OUTPUT_MODEL_PATH    = "output_malayalam_lora"                       # where checkpoints go
TRAIN_JSONL          = "malayalam_data/train_with_codes.jsonl"
VAL_JSONL            = "malayalam_data/val_with_codes.jsonl"         # set to None to skip

# ─── Training hparams ────────────────────────────────────────────────────────
BATCH_SIZE                  = 1        # keep at 1 for 4 GB VRAM
EVAL_BATCH_SIZE             = 1
GRAD_ACCUMULATION_STEPS     = 8        # effective batch = 8
LR                          = 2e-6
NUM_EPOCHS                  = 10
START_EPOCH                 = 0        # >0 only when resuming
SAVE_EVERY                  = 1        # save checkpoint every N epochs
EVAL_EVERY                  = 1        # evaluate every N epochs
MIXED_PRECISION             = "bf16"   # "no", "fp16", or "bf16"
ATTN_IMPLEMENTATION         = "sdpa"   # "sdpa" or "flash_attention_2"
SPEAKER_NAME                = "malayalam_speaker"

# ─── LoRA hparams ─────────────────────────────────────────────────────────────
LORA_RANK           = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_BIAS           = "none"           # "none", "all", or "lora_only"
LORA_TARGET_MODULES = "q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj"

# ─── Resume from existing adapter? ───────────────────────────────────────────
RESUME_ADAPTER = None                  # e.g. "output_malayalam_lora/checkpoint-epoch-2"

print("Configuration loaded.")

## 2. Imports

In [ ]:
import json
import torch
from accelerate import Accelerator
from dataset import TTSDataset
from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel
from safetensors.torch import save_file
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoConfig

try:
    from peft import LoraConfig, TaskType, get_peft_model, PeftModel
except ImportError as exc:
    raise SystemExit(
        "peft is required for LoRA fine-tuning. Install it with: pip install peft"
    ) from exc

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

## 3. Helper Functions

In [ ]:
target_speaker_embedding = None


def _parse_list(value: str) -> list[str]:
    return [item.strip() for item in value.split(",") if item.strip()]


def compute_loss(model, batch):
    """Forward pass + loss computation for one batch."""
    global target_speaker_embedding

    input_ids           = batch["input_ids"]
    codec_ids           = batch["codec_ids"]
    ref_mels            = batch["ref_mels"]
    text_embedding_mask = batch["text_embedding_mask"]
    codec_embedding_mask= batch["codec_embedding_mask"]
    attention_mask      = batch["attention_mask"]
    codec_0_labels      = batch["codec_0_labels"]
    codec_mask          = batch["codec_mask"]

    with torch.no_grad():
        speaker_embedding = model.speaker_encoder(
            ref_mels.to(
                dtype=next(model.parameters()).dtype,
                device=next(model.parameters()).device
            )
        ).detach()

    if model.training and target_speaker_embedding is None:
        target_speaker_embedding = speaker_embedding.detach().to("cpu")

    input_text_ids  = input_ids[:, :, 0]
    input_codec_ids = input_ids[:, :, 1]

    input_text_embedding = model.talker.model.text_embedding(input_text_ids)
    if hasattr(model.talker, "text_projection"):
        input_text_embedding = model.talker.text_projection(input_text_embedding)
    input_text_embedding  = input_text_embedding * text_embedding_mask
    input_codec_embedding = model.talker.model.codec_embedding(input_codec_ids) * codec_embedding_mask
    input_codec_embedding[:, 6, :] = speaker_embedding

    input_embeddings = input_text_embedding + input_codec_embedding

    for i in range(1, 16):
        codec_i_embedding  = model.talker.code_predictor.get_input_embeddings()[i - 1](codec_ids[:, :, i])
        codec_i_embedding  = codec_i_embedding * codec_mask.unsqueeze(-1)
        input_embeddings  += codec_i_embedding

    outputs = model.talker(
        inputs_embeds=input_embeddings[:, :-1, :],
        attention_mask=attention_mask[:, :-1],
        labels=None,          # manual loss to avoid HF double-shift bug
        output_hidden_states=True,
    )

    # Codec-0 loss (manually shifted)
    logits           = outputs.logits            # [B, T-1, V]
    codec_0_targets  = codec_0_labels[:, 1:]     # already shifted
    codec_0_loss = torch.nn.functional.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        codec_0_targets.reshape(-1),
        ignore_index=-100,
    )

    hidden_states       = outputs.hidden_states[0][-1]
    talker_hidden_states = hidden_states[codec_mask[:, 1:]]
    talker_codec_ids     = codec_ids[codec_mask]

    _, sub_talker_loss = model.talker.forward_sub_talker_finetune(
        talker_codec_ids, talker_hidden_states
    )

    return codec_0_loss + sub_talker_loss


def evaluate(model, dataloader, accelerator):
    """Compute mean validation loss."""
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in dataloader:
            loss    = compute_loss(model, batch)
            gathered = accelerator.gather_for_metrics(loss.detach())
            if gathered.ndim == 0:
                gathered = gathered.unsqueeze(0)
            losses.append(gathered)
    if not losses:
        return None
    return torch.cat(losses).mean().item()


print("Helper functions defined.")

## 4. Setup Accelerator

In [ ]:
os.makedirs(OUTPUT_MODEL_PATH, exist_ok=True)

accelerator = Accelerator(
    gradient_accumulation_steps=GRAD_ACCUMULATION_STEPS,
    mixed_precision=None if MIXED_PRECISION == "no" else MIXED_PRECISION,
    log_with="tensorboard",
    project_dir=OUTPUT_MODEL_PATH,
)

print(f"Device         : {accelerator.device}")
print(f"Mixed precision: {MIXED_PRECISION}")
print(f"Grad accum     : {GRAD_ACCUMULATION_STEPS}")

## 5. Load Base Model

In [ ]:
print(f"Loading model from: {INIT_MODEL_PATH}")

qwen3tts = Qwen3TTSModel.from_pretrained(
    INIT_MODEL_PATH,
    torch_dtype=torch.bfloat16,
    attn_implementation=ATTN_IMPLEMENTATION,
)
config = AutoConfig.from_pretrained(INIT_MODEL_PATH)

print("Base model loaded.")

## 6. Apply LoRA Adapter

In [ ]:
if RESUME_ADAPTER:
    print(f"Resuming from adapter: {RESUME_ADAPTER}")
    model = PeftModel.from_pretrained(
        qwen3tts.model,
        RESUME_ADAPTER,
        is_trainable=True,
    )
else:
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias=LORA_BIAS,
        target_modules=_parse_list(LORA_TARGET_MODULES),
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(qwen3tts.model, lora_config)

# use_reentrant=False is required: reentrant gradient checkpointing severs the
# autograd graph for LoRA adapters when no raw inputs have requires_grad=True.
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

model.print_trainable_parameters()

## 7. Build Dataloaders

In [ ]:
with open(TRAIN_JSONL, "r", encoding="utf-8") as f:
    train_data = [json.loads(line) for line in f]

dataset        = TTSDataset(train_data, qwen3tts.processor, config)
train_dataloader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=dataset.collate_fn
)
print(f"Training samples : {len(train_data)}")

val_dataloader = None
if VAL_JSONL:
    with open(VAL_JSONL, "r", encoding="utf-8") as f:
        val_data = [json.loads(line) for line in f]
    val_dataset    = TTSDataset(val_data, qwen3tts.processor, config)
    eval_bs        = EVAL_BATCH_SIZE or BATCH_SIZE
    val_dataloader = DataLoader(
        val_dataset, batch_size=eval_bs, shuffle=False, collate_fn=val_dataset.collate_fn
    )
    print(f"Validation samples: {len(val_data)}")
else:
    print("No validation set.")

## 8. Prepare Optimizer & Accelerate

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

if val_dataloader is not None:
    model, optimizer, train_dataloader, val_dataloader = accelerator.prepare(
        model, optimizer, train_dataloader, val_dataloader
    )
else:
    model, optimizer, train_dataloader = accelerator.prepare(
        model, optimizer, train_dataloader
    )

print("Optimizer and dataloaders prepared.")

## 9. Training Loop

In [ ]:
target_speaker_embedding = None  # reset before training
model.train()

for local_epoch in range(NUM_EPOCHS):
    epoch = START_EPOCH + local_epoch

    for step, batch in enumerate(train_dataloader):
        with accelerator.accumulate(model):
            loss = compute_loss(model, batch)
            accelerator.backward(loss)

            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            optimizer.zero_grad()

        if step % 10 == 0:
            accelerator.print(f"Epoch {epoch} | Step {step} | Loss: {loss.item():.4f}")

    # ── Validation ────────────────────────────────────────────────────────────
    if val_dataloader is not None and (local_epoch + 1) % EVAL_EVERY == 0:
        val_loss = evaluate(model, val_dataloader, accelerator)
        if accelerator.is_main_process and val_loss is not None:
            accelerator.print(f"Epoch {epoch} | Val Loss: {val_loss:.4f}")
        model.train()

    # ── Save checkpoint ───────────────────────────────────────────────────────
    if accelerator.is_main_process and (local_epoch + 1) % SAVE_EVERY == 0:
        output_dir = os.path.join(OUTPUT_MODEL_PATH, f"checkpoint-epoch-{epoch}")
        os.makedirs(output_dir, exist_ok=True)

        unwrapped_model = accelerator.unwrap_model(model)
        unwrapped_model.save_pretrained(output_dir, safe_serialization=True)

        # Patch config.json with speaker metadata
        input_config_file  = os.path.join(INIT_MODEL_PATH, "config.json")
        output_config_file = os.path.join(output_dir, "config.json")
        with open(input_config_file, "r", encoding="utf-8") as f:
            config_dict = json.load(f)
        config_dict["tts_model_type"] = "custom_voice"
        talker_config = config_dict.get("talker_config", {})
        talker_config.setdefault("spk_id", {})
        talker_config.setdefault("spk_is_dialect", {})
        talker_config["spk_id"][SPEAKER_NAME]       = 3000
        talker_config["spk_is_dialect"][SPEAKER_NAME] = False
        config_dict["talker_config"] = talker_config
        with open(output_config_file, "w", encoding="utf-8") as f:
            json.dump(config_dict, f, indent=2, ensure_ascii=False)

        # Save the captured speaker embedding
        if target_speaker_embedding is not None:
            save_file(
                {"target_speaker_embedding": target_speaker_embedding[0]},
                os.path.join(output_dir, "speaker_embedding.safetensors"),
            )

        print(f"✓ Checkpoint saved → {output_dir}")

print("\n✅ Training complete!")

## 10. Quick Inference Test

Edit `CHECKPOINT_DIR` to point to the checkpoint you want to test.

In [ ]:
import soundfile as sf
import IPython.display as ipd

CHECKPOINT_DIR  = os.path.join(OUTPUT_MODEL_PATH, "checkpoint-epoch-0")
REF_AUDIO       = "malayalam_data/wavs/sample_0001.wav"   # reference audio for voice cloning
TEST_TEXT       = "ഈ ഒരു മലയാള ടെക്സ്ററ്റ് ടു സ്പീച്ച് സിസ്റ്റം ആണ്."
OUTPUT_WAV      = "test_output.wav"

# Load fine-tuned model
from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel
from peft import PeftModel

base = Qwen3TTSModel.from_pretrained(
    INIT_MODEL_PATH,
    torch_dtype=torch.bfloat16,
    attn_implementation=ATTN_IMPLEMENTATION,
)
base.model = PeftModel.from_pretrained(base.model, CHECKPOINT_DIR)

result_wavs, sr = base.generate_voice_clone(
    text=TEST_TEXT,
    ref_audio=REF_AUDIO,
    x_vector_only_mode=True,
)

sf.write(OUTPUT_WAV, result_wavs[0], sr)
print(f"Saved to {OUTPUT_WAV}")
ipd.Audio(OUTPUT_WAV)